# Basile engine 3 · The smaller published pieces
Basile's public repositories, checked 2026-09-30: **libraryofbabel.info-algo** (the Babelia PRNG, its search, and `Euclid's Extended`; CC BY-SA-NC) · **LoB-API** (a CC0 licence file only; no code) · **LobThis** and **LoBThis-Firefox** (browser extensions; LobThis is a fork of `rik-degraaff/LobThis`, licence not stated) · **newsometimesatdawn**, **Dezmediah-site** (web sites).

In [1]:
import sys, os, importlib.util, math, time, random
sys.path.insert(0, os.path.abspath('../..'))
def load_maths():
    spec = importlib.util.spec_from_file_location("basile_maths", "../../modules/basile/maths.py")
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
M = load_maths()
CACHE = M.default_cache_dir()
HAVE = os.path.exists(os.path.join(CACHE, "PRNG")) and os.path.exists(os.path.join(CACHE, "imagesearch.cpp"))
print("engine loaded; published files in cache:", HAVE, "| gmpy2 accelerator:", M._gmpy2 is not None)

engine loaded; published files in cache: True | gmpy2 accelerator: True


## `Euclid's Extended` (lcgreverse.cpp): a template with a 25-entry array
The file's `m` and `a` are empty strings (`mpz_int m("")`), so it is a template. It keeps the running coefficients in `qarray[25]` and writes `qarray[i]` for `i = 2, 3, …`: it holds 23 divisions. The port raises `OverflowError` where the C++ would write past the array.

In [2]:
print("3^-1 mod 7 =", M.euclid_inverse_qarray(3, 7), "| 17^-1 mod 3120 =", M.euclid_inverse_qarray(17, 3120), "(divisions:", M.euclid_steps(17, 3120), ")")
rnd = random.Random(2); over = 0; steps = []
for _ in range(200):
    m = rnd.getrandbits(64) | (1 << 63) | 1; a = rnd.getrandbits(63) | 1
    steps.append(M.euclid_steps(a, m))
    try: M.euclid_inverse_qarray(a, m)
    except OverflowError: over += 1
print("random 64-bit inputs: overflow in %d of 200 | mean divisions %.1f | the array allows %d" % (over, sum(steps) / 200, M.EUCLID_QARRAY_SIZE - 2))
a_bits = 1597519
print("published a has %d bits: about 0.843*ln(a) = %.0f divisions (ESTIMATE), against 23 in the array" % (a_bits, 0.843 * a_bits * math.log(2)))
assert over > 150

3^-1 mod 7 = 5 | 17^-1 mod 3120 = 2753 (divisions: 3 )
random 64-bit inputs: overflow in 199 of 200 | mean divisions 36.0 | the array allows 23
published a has 1597519 bits: about 0.843*ln(a) = 933467 divisions (ESTIMATE), against 23 in the array


`pow(a, -1, m)` (used by the engine) has no such limit.

## LobThis: one request, built as data
The extension submits the current page's HTML as the `extension` field of a POST form to `resourcelocator.cgi`. The engine returns that request and sends nothing.

In [3]:
print(M.resource_locator_request("some page text"))

{'method': 'POST', 'url': 'http://libraryofbabel.info/resourcelocator.cgi', 'fields': {'extension': 'some page text'}}


## Site addresses
`<hexagon>-w<wall>-s<shelf>-v<volume>[:<page>]`; per the site's Reference Hex page: 4 walls, 5 shelves per wall, 32 volumes per shelf, 410 pages per volume, 40 lines of about 80 characters.

In [4]:
r = M.parse_babel_address("jeb0110jlb-w2-s4-v16:19"); print(r)
print("volumes per hexagon:", M.WALLS_PER_HEXAGON * M.SHELVES_PER_WALL * M.BOOKS_PER_SHELF, "| characters per volume:", M.PAGES_PER_BOOK * M.LINES_PER_PAGE * M.CHARS_PER_LINE)
print("volume index is the last volume of the last shelf of the last wall:", M.parse_babel_address("x-w4-s5-v32")["volume_index"])

{'hexagon': 'jeb0110jlb', 'wall': 2, 'shelf': 4, 'volume': 16, 'page': 19, 'volume_index': 271}
volumes per hexagon: 640 | characters per volume: 1312000
volume index is the last volume of the last shelf of the last wall: 639


## The scale of the text library, recomputed

In [5]:
s = M.library_scale(); print(s)
assert round(s["log10_books"]) == 4677

{'log10_pages': 4679.67359327666, 'log10_books': 4677.06080941994, 'log10_full_books': 1918666.1732434304}
